# OpenML Tuning

OpenML 9개 태스크에서 AutoGluon GPU 설정의 실제 성능을 직접 측정하고 정리하는 단일 노트북입니다.

이 노트북에서 바로 하는 일:
- `gpu_default` 설정으로 AutoGluon 성능 측정
- 다른 모델과 비교할 기준 결과 정리
- 실행 중 발생한 문제와 해결 방법 정리
- 한글 요약 문구 자동 생성

고정 조건:
- Kaggle T4
- `time_limit=600`
- `train 80% / test 20%`
- AutoGluon Tabular `1.5.x`
- 기본 GPU 수 `1`

주의:
- 이번 결과는 holdout 실험이므로 OpenML 공식 fold 및 논문 점수와 1:1 비교용은 아닙니다.
- AutoGluon Tabular는 GPU 모델과 CPU 모델이 섞여 돌아갈 수 있습니다.


## 1. Runtime Check

이 구간은 실험 환경을 확인하는 단계입니다.

확인하는 내용:
- Kaggle 런타임인지
- GPU가 잡혔는지
- 결과 저장 디렉터리와 OpenML 캐시 위치가 어디인지


In [12]:
from pathlib import Path
import platform
import subprocess
import sys
import warnings

warnings.filterwarnings('ignore')

print('Python:', sys.version)
subprocess.run(['bash', '-lc', 'nvidia-smi -L || true'], check=False)

IS_KAGGLE = Path('/kaggle').exists()
WORK_DIR = Path('/kaggle/working') if IS_KAGGLE else Path.cwd()
RESULTS_DIR = WORK_DIR / 'autogluon_openml_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OPENML_CACHE_DIR = WORK_DIR / 'openml_cache'
OPENML_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('Platform:', platform.platform())
print('IS_KAGGLE:', IS_KAGGLE)
print('WORK_DIR:', WORK_DIR)
print('RESULTS_DIR:', RESULTS_DIR)
print('OPENML_CACHE_DIR:', OPENML_CACHE_DIR)


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
GPU 0: Tesla T4 (UUID: GPU-04a948a2-d4ae-717a-81e4-b67e63c379a5)
GPU 1: Tesla T4 (UUID: GPU-ec5f1f8d-dfbf-4143-cbb1-239d4627b51c)
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
IS_KAGGLE: True
WORK_DIR: /kaggle/working
RESULTS_DIR: /kaggle/working/autogluon_openml_results
OPENML_CACHE_DIR: /kaggle/working/openml_cache


## 2. Package Setup

이 구간은 Kaggle 기본 환경에 필요한 패키지를 맞추는 단계입니다.

구성:
- `autogluon.tabular`, `openml`, `pandas`만 설치해서 실험 흐름을 최대한 단순하게 유지합니다.
- 벤치마크 프레임워크 전체를 설치하는 대신, 실제 실험에 필요한 최소 구성만 사용합니다.


In [13]:
import subprocess
import sys

def run_cmd(args):
    print('$', ' '.join(args))
    subprocess.run(args, check=True)

run_cmd([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
run_cmd([
    sys.executable,
    '-m',
    'pip',
    'install',
    'autogluon.tabular[all]>=1.5,<1.6',
    'openml',
    'pandas',
])

print('Install complete.')


$ /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
$ /usr/bin/python3 -m pip install autogluon.tabular[all]>=1.5,<1.6 openml pandas
Install complete.


## 3. Experiment Design

여기서는 비교할 실험 조건을 고정합니다.

핵심 설정:
- OpenML 9개 태스크만 사용
- `time_limit=600`, `train_ratio=0.8`, `split_seed=42`
- 실험 설정: `gpu_default`

설정 의미:
- `gpu_default`: AutoGluon 기본 탐색 공간을 유지하면서 GPU만 켠 대표 설정
- 다른 모델과 비교할 때 AutoGluon 쪽 기준 결과로 사용


In [14]:
import json
import pandas as pd

TASKS = [
    {'task_order': 10, 'name': 'guillermo', 'openml_id': 168337, 'metric': 'roc_auc'},
    {'task_order': 11, 'name': 'riccardo', 'openml_id': 168338, 'metric': 'roc_auc'},
    {'task_order': 12, 'name': 'amazon_employee', 'openml_id': 34539, 'metric': 'roc_auc'},
    {'task_order': 13, 'name': 'nomao', 'openml_id': 9977, 'metric': 'roc_auc'},
    {'task_order': 14, 'name': 'bank-marketing', 'openml_id': 14965, 'metric': 'roc_auc'},
    {'task_order': 15, 'name': 'adult', 'openml_id': 7592, 'metric': 'roc_auc'},
    {'task_order': 16, 'name': 'kddcup09_appetency', 'openml_id': 3945, 'metric': 'roc_auc'},
    {'task_order': 17, 'name': 'apsfailure', 'openml_id': 168868, 'metric': 'roc_auc'},
    {'task_order': 18, 'name': 'numerai28.6', 'openml_id': 167120, 'metric': 'roc_auc'},
]

EXPERIMENT_NAME = 'openml_gpu_default'
TIME_LIMIT = 600
TRAIN_RATIO = 0.8
SPLIT_SEED = 42
VERBOSITY = 2
NUM_GPUS = 1
MAX_TASKS = None
RESUME = True
DRY_RUN = False

RECIPE_LIBRARY = {
    'gpu_default': {
        'description': 'GPU enabled, default search space',
        'num_gpus': NUM_GPUS,
        'presets': None,
    },
}
ACTIVE_RECIPE_NAMES = ['gpu_default']
BASELINE_RECIPE_NAME = None

selected_tasks = TASKS[:MAX_TASKS] if MAX_TASKS else TASKS
active_recipes = []
for recipe_name in ACTIVE_RECIPE_NAMES:
    recipe_cfg = RECIPE_LIBRARY[recipe_name].copy()
    recipe_cfg['recipe_name'] = recipe_name
    active_recipes.append(recipe_cfg)

RESULTS_CSV = RESULTS_DIR / f'{EXPERIMENT_NAME}_results.csv'
RUN_CONFIG_PATH = RESULTS_DIR / f'{EXPERIMENT_NAME}_config.json'

existing_results = pd.DataFrame()
completed_keys = set()
if RESUME and RESULTS_CSV.exists():
    existing_results = pd.read_csv(RESULTS_CSV)
    required_cols = {'dataset', 'recipe_name'}
    if required_cols.issubset(existing_results.columns):
        completed_keys = set(zip(existing_results['dataset'].astype(str), existing_results['recipe_name'].astype(str)))

run_config = {
    'experiment_name': EXPERIMENT_NAME,
    'tasks': selected_tasks,
    'active_recipe_names': ACTIVE_RECIPE_NAMES,
    'time_limit': TIME_LIMIT,
    'train_ratio': TRAIN_RATIO,
    'split_seed': SPLIT_SEED,
    'num_gpus': NUM_GPUS,
    'results_csv': str(RESULTS_CSV),
}
RUN_CONFIG_PATH.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding='utf-8')

print('selected task count:', len(selected_tasks))
print('selected task names:', [task['name'] for task in selected_tasks])
print('ACTIVE_RECIPE_NAMES:', ACTIVE_RECIPE_NAMES)
print('TIME_LIMIT:', TIME_LIMIT)
print('TRAIN_RATIO:', TRAIN_RATIO)
print('NUM_GPUS:', NUM_GPUS)
print('RESULTS_CSV:', RESULTS_CSV)
print('RUN_CONFIG_PATH:', RUN_CONFIG_PATH)
print('already completed:', len(completed_keys))

pd.DataFrame(selected_tasks)


selected task count: 9
selected task names: ['guillermo', 'riccardo', 'amazon_employee', 'nomao', 'bank-marketing', 'adult', 'kddcup09_appetency', 'apsfailure', 'numerai28.6']
ACTIVE_RECIPE_NAMES: ['gpu_default']
TIME_LIMIT: 600
TRAIN_RATIO: 0.8
NUM_GPUS: 1
RESULTS_CSV: /kaggle/working/autogluon_openml_results/openml_gpu_default_results.csv
RUN_CONFIG_PATH: /kaggle/working/autogluon_openml_results/openml_gpu_default_config.json
already completed: 0


,task_order,name,openml_id,metric
0,10,guillermo,168337,roc_auc
1,11,riccardo,168338,roc_auc
2,12,amazon_employee,34539,roc_auc
3,13,nomao,9977,roc_auc
4,14,bank-marketing,14965,roc_auc
5,15,adult,7592,roc_auc
6,16,kddcup09_appetency,3945,roc_auc
7,17,apsfailure,168868,roc_auc
8,18,numerai28.6,167120,roc_auc


## 4. Helper Functions

이 구간은 실험을 반복 가능하게 만드는 공통 함수들입니다.

포함된 역할:
- OpenML 데이터셋 다운로드 및 로드
- `80:20` stratified holdout 분할
- AutoGluon 입력 형태로 데이터 변환
- AutoGluon 학습 설정 구성
- ROC AUC 계산 및 단일 task 실행 함수 정의


In [15]:
import shutil
import tempfile
import time

import openml
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

openml.config.cache_directory = str(OPENML_CACHE_DIR)

def load_task_dataset(task_id):
    task = openml.tasks.get_task(task_id, download_splits=True)
    dataset = task.get_dataset()
    X, y, _, _ = dataset.get_data(
        dataset_format='dataframe',
        target=dataset.default_target_attribute,
    )
    target_col = dataset.default_target_attribute
    if not isinstance(y, pd.Series):
        y = pd.Series(y, name=target_col)
    return X.copy(), y.copy(), target_col

def split_task_data(task_id, train_ratio, split_seed):
    X, y, target_col = load_task_dataset(task_id)
    label_counts = y.value_counts(dropna=False)
    stratify = y if len(label_counts) > 1 and int(label_counts.min()) >= 2 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        train_size=train_ratio,
        random_state=split_seed,
        shuffle=True,
        stratify=stratify,
    )
    return X_train.copy(), X_test.copy(), y_train.copy(), y_test.copy(), target_col

def make_df(X, y, target_col):
    df = X.copy()
    df[target_col] = y.to_numpy()
    return df

def score_task(predictor, X_test, y_test):
    y_proba = predictor.predict_proba(X_test)
    if isinstance(y_proba, pd.DataFrame):
        if y_proba.shape[1] < 2:
            raise ValueError('ROC AUC scoring requires binary probability columns.')
        positive_proba = y_proba.iloc[:, 1]
    else:
        positive_proba = y_proba
    return float(roc_auc_score(y_test, positive_proba))

def run_single_experiment(task_cfg, recipe_cfg):
    task_name = task_cfg['name']
    recipe_name = recipe_cfg['recipe_name']
    model_dir = Path(tempfile.mkdtemp(prefix=f'{task_name}_{recipe_name}_', dir=str(WORK_DIR)))
    started = time.time()

    row = {
        'task_order': task_cfg['task_order'],
        'dataset': task_name,
        'openml_id': task_cfg['openml_id'],
        'metric': task_cfg['metric'],
        'recipe_name': recipe_name,
        'score': None,
        'train_sec': None,
        'best_model': None,
        'trained_models': None,
        'status': 'error',
        'error': None,
    }

    try:
        X_train, X_test, y_train, y_test, target_col = split_task_data(
            task_cfg['openml_id'],
            train_ratio=TRAIN_RATIO,
            split_seed=SPLIT_SEED,
        )
        train_df = make_df(X_train, y_train, target_col)

        predictor = TabularPredictor(
            label=target_col,
            eval_metric=task_cfg['metric'],
            path=str(model_dir),
            verbosity=VERBOSITY,
        )

        fit_kwargs = {
            'train_data': train_df,
            'time_limit': TIME_LIMIT,
            'num_gpus': recipe_cfg['num_gpus'],
        }
        if recipe_cfg['presets'] is not None:
            fit_kwargs['presets'] = recipe_cfg['presets']

        predictor.fit(**fit_kwargs)
        leaderboard = predictor.leaderboard(silent=True)

        row['score'] = score_task(predictor, X_test, y_test)
        row['train_sec'] = round(time.time() - started, 2)
        row['best_model'] = predictor.model_best
        row['trained_models'] = int(len(leaderboard))
        row['status'] = 'ok'
        return row
    except Exception as exc:
        row['train_sec'] = round(time.time() - started, 2)
        row['error'] = str(exc)
        return row
    finally:
        shutil.rmtree(model_dir, ignore_errors=True)


## 5. Run Benchmark

이 구간은 태스크와 recipe를 순차적으로 돌리면서 결과를 누적 저장하는 실행 셀입니다.

실행 방식:
- 각 데이터셋에 대해 `gpu_default` 설정으로 실행
- 매 실행마다 결과를 CSV에 저장
- 이미 끝난 조합은 `RESUME=True`일 때 자동으로 건너뜀


In [16]:
import pandas as pd
import time

run_rows = []
if not existing_results.empty:
    run_rows.extend(existing_results.to_dict(orient='records'))

if DRY_RUN:
    print('DRY_RUN=True 이므로 실제 실행은 건너뜁니다.')
else:
    session_start = time.time()
    total_runs = len(selected_tasks) * len(active_recipes)
    current_run = 0

    for task_cfg in selected_tasks:
        for recipe_cfg in active_recipes:
            current_run += 1
            run_key = (task_cfg['name'], recipe_cfg['recipe_name'])
            if run_key in completed_keys:
                print(f"[{current_run}/{total_runs}] skip: {task_cfg['name']} / {recipe_cfg['recipe_name']} (already completed)")
                continue

            elapsed_min = (time.time() - session_start) / 60
            print(f"\n[{current_run}/{total_runs}] elapsed={elapsed_min:.1f} min")
            row = run_single_experiment(task_cfg, recipe_cfg)
            print({k: row[k] for k in ['dataset', 'recipe_name', 'score', 'train_sec', 'status', 'error']})
            run_rows.append(row)
            completed_keys.add(run_key)
            pd.DataFrame(run_rows).sort_values(['task_order', 'recipe_name']).to_csv(RESULTS_CSV, index=False)

    print(f"\nDone. Total elapsed: {(time.time() - session_start) / 60:.1f} min")

run_df = pd.DataFrame(run_rows)
if not run_df.empty:
    run_df = run_df.sort_values(['task_order', 'recipe_name']).reset_index(drop=True)

print('saved to:', RESULTS_CSV)
print('rows:', len(run_df))
run_df.tail()



[1/9] elapsed=0.0 min


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Sat Jan 17 11:20:45 UTC 2026
CPU Count:          4
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.54/14.56 GB | GPU 1: 14.56/14.56 GB
Total GPU Memory:   Free: 29.10 GB, Allocated: 0.03 GB, Total: 29.13 GB
GPU Count:          2
Memory Avail:       18.29 GB / 31.35 GB (58.3%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabul

[1000]	valid_set's binary_logloss: 0.415974


	0.8819	 = Validation score   (roc_auc)
	182.0s	 = Training   runtime
	0.12s	 = Validation runtime
Fitting model: LightGBM ... Training model for up to 405.85s of the 405.85s of remaining time.
	Fitting with cpus=2, gpus=1, mem=3.3/17.3 GB
	0.8855	 = Validation score   (roc_auc)
	169.55s	 = Training   runtime
	0.09s	 = Validation runtime
Fitting model: RandomForestGini ... Training model for up to 236.18s of the 236.18s of remaining time.
	Fitting with cpus=4, gpus=0, mem=0.1/17.3 GB
	0.8674	 = Validation score   (roc_auc)
	79.25s	 = Training   runtime
	0.14s	 = Validation runtime
Fitting model: RandomForestEntr ... Training model for up to 156.73s of the 156.73s of remaining time.
	Fitting with cpus=4, gpus=0, mem=0.1/17.3 GB
	0.8713	 = Validation score   (roc_auc)
	80.63s	 = Training   runtime
	0.14s	 = Validation runtime
Fitting model: CatBoost ... Training model for up to 75.90s of the 75.89s of remaining time.
	Fitting with cpus=2, gpus=1, mem=6.6/17.3 GB
	Training CatBoost with G

{'dataset': 'guillermo', 'recipe_name': 'gpu_default', 'score': 0.9057565886681392, 'train_sec': 611.62, 'status': 'ok', 'error': None}

[2/9] elapsed=10.2 min


...

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Sat Jan 17 11:20:45 UTC 2026
CPU Count:          4
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.54/14.56 GB | GPU 1: 14.56/14.56 GB
Total GPU Memory:   Free: 29.10 GB, Allocated: 0.03 GB, Total: 29.13 GB
GPU Count:          2
Memory Avail:       19.52 GB / 31.35 GB (62.3%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabul

{'dataset': 'apsfailure', 'recipe_name': 'gpu_default', 'score': 0.9961735952489721, 'train_sec': 298.76, 'status': 'ok', 'error': None}

[9/9] elapsed=38.7 min


Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/kaggle/working/numerai28.6_gpu_default_j16v3m4l"
Train Data Rows:    77056
Train Data Columns: 21
Label Column:       attribute_21
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  ['1', '0']
	If 'binary' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
	Note: For your binary classification, AutoGluon arbitrarily selected which label-value represents positive (1) vs negative (0) class.
	To explicitly set the positive_class, either rename classes to 1 and 0, or specify positive_class in Predictor init.
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipel

{'dataset': 'numerai28.6', 'recipe_name': 'gpu_default', 'score': 0.522821466325643, 'train_sec': 252.54, 'status': 'ok', 'error': None}

Done. Total elapsed: 42.9 min
saved to: /kaggle/working/autogluon_openml_results/openml_gpu_default_results.csv
rows: 9


,task_order,dataset,openml_id,metric,recipe_name,score,train_sec,best_model,trained_models,status,error
4,14,bank-marketing,14965,roc_auc,gpu_default,0.936927,167.88,WeightedEnsemble_L2,12,ok,None
5,15,adult,7592,roc_auc,gpu_default,0.931600,175.66,WeightedEnsemble_L2,12,ok,None
6,16,kddcup09_appetency,3945,roc_auc,gpu_default,0.859398,199.70,WeightedEnsemble_L2,12,ok,None
7,17,apsfailure,168868,roc_auc,gpu_default,0.996174,298.76,WeightedEnsemble_L2,12,ok,None
8,18,numerai28.6,167120,roc_auc,gpu_default,0.522821,252.54,WeightedEnsemble_L2,12,ok,None


## 6. Known Issues

| 이슈 | 원인 | 정리 |
| --- | --- | --- |
| 평가 프로토콜 차이 | 이번 노트북은 `train 80% / test 20%` holdout을 사용함 | 논문/OpenML 공식 fold 점수와 직접 비교가 아니라 기준 결과로 해석 |
| GPU를 켜도 일부 단계는 CPU로 실행될 수 있음 | AutoGluon Tabular 기본 탐색 공간이 CPU/GPU 혼합 모델군이기 때문 | `gpu_default`를 GPU 활성화 대표 설정으로 해석 |
| GPU 2개를 항상 꽉 채워 쓰기 어려움 | Tabular fit이 사실상 순차적으로 진행되고 개별 모델도 대개 1개 GPU를 사용함 | 이번 발표는 비교 단순화를 위해 `num_gpus=1` 기준으로 설명 |

실제 실행 중 발생한 실패는 아래 결과 요약 셀에서 별도로 자동 표시합니다.


## 7. Result Interpretation

이 구간은 실험 결과를 다시 정리하는 셀입니다.

자동으로 나오는 내용:
- raw result table
- task별 결과 표
- 평균 점수와 평균 학습 시간
- 실제 실패 유형 분류
- 한글 요약 문단


In [17]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

def classify_issue(error_text):
    if pd.isna(error_text) or not str(error_text).strip():
        return None
    text = str(error_text).lower()
    if 'bin size' in text and 'gpu' in text:
        return 'LightGBM GPU 제한'
    if 'cuda' in text or 'gpu' in text:
        return 'GPU 설정/호환 이슈'
    if 'out of memory' in text or 'memoryerror' in text or 'std::bad_alloc' in text:
        return '메모리 부족'
    if 'roc auc' in text or 'probability columns' in text or 'log_loss' in text:
        return '평가 지표/라벨 이슈'
    if 'openml' in text or 'http' in text or 'connection' in text or 'download' in text:
        return '데이터 다운로드/네트워크 이슈'
    return '기타 실행 이슈'

if not RESULTS_CSV.exists():
    print('results.csv 가 아직 없습니다. 실행 셀 로그를 확인하세요.')
else:
    results_df = pd.read_csv(RESULTS_CSV).sort_values(['task_order', 'recipe_name']).reset_index(drop=True)
    results_df['error'] = results_df['error'].fillna('').astype(str).str.strip()
    results_df['issue_type'] = results_df['error'].apply(classify_issue)

    recipe_meta_df = pd.DataFrame(active_recipes)[['recipe_name', 'description', 'num_gpus']]
    recipe_meta_df = recipe_meta_df.rename(columns={'description': 'recipe_description'})
    results_df = results_df.drop(columns=['recipe_description', 'num_gpus'], errors='ignore')
    results_df = results_df.merge(recipe_meta_df, on='recipe_name', how='left')

    print('Raw experiment results')
    display_cols = [
        'task_order',
        'dataset',
        'recipe_name',
        'recipe_description',
        'score',
        'train_sec',
        'best_model',
        'trained_models',
        'status',
        'issue_type',
        'error',
    ]
    display(results_df[[c for c in display_cols if c in results_df.columns]])

    failure_df = results_df[results_df['status'] != 'ok'].copy()
    ok_df = results_df[results_df['status'] == 'ok'].copy()

    if ok_df.empty:
        print('성공한 실행이 아직 없습니다.')
        recipe_summary = pd.DataFrame()
        score_pivot = pd.DataFrame()
    else:
        score_pivot = ok_df.pivot_table(index='dataset', columns='recipe_name', values='score')
        time_pivot = ok_df.pivot_table(index='dataset', columns='recipe_name', values='train_sec')

        recipe_summary = results_df.groupby(['recipe_name', 'recipe_description'], as_index=False).agg(
            planned_runs=('dataset', 'size'),
            success_runs=('status', lambda s: int((s == 'ok').sum())),
            failed_runs=('status', lambda s: int((s != 'ok').sum())),
            mean_score=('score', 'mean'),
            median_score=('score', 'median'),
            mean_train_sec=('train_sec', 'mean'),
            median_train_sec=('train_sec', 'median'),
            mean_trained_models=('trained_models', 'mean'),
        )
        recipe_summary['success_rate'] = (recipe_summary['success_runs'] / recipe_summary['planned_runs']).round(3)
        recipe_summary = recipe_summary.sort_values(['mean_score'], ascending=[False], na_position='last').reset_index(drop=True)

        print('Recipe summary for presentation')
        display(recipe_summary)

        print('Score pivot')
        display(score_pivot)

    if failure_df.empty:
        print('Observed failures: none')
    else:
        print('Observed failures')
        display(failure_df[['task_order', 'dataset', 'recipe_name', 'issue_type', 'error']])

    presentation_lines = [
        '## Presentation Draft',
        '- 이번 실험은 OpenML 9개 태스크를 Kaggle T4, 600초, train 80% / test 20% 조건에서 실행한 결과입니다.',
        '- 이번 노트북은 AutoGluon 내부 recipe 비교 대신 `gpu_default` 설정 하나를 기준 결과로 사용합니다.',
    ]

    if ok_df.empty:
        presentation_lines.append('- 아직 성공한 실행이 없어서 먼저 실행 셀을 완료한 뒤 요약을 확인해야 합니다.')
    else:
        best_score_row = recipe_summary.dropna(subset=['mean_score']).iloc[0]
        presentation_lines.append(
            f"- 평균 점수 기준으로는 `{best_score_row['recipe_name']}`가 가장 높았고, 평균 점수는 {best_score_row['mean_score']:.4f}였습니다."
        )

        fastest_row = recipe_summary.dropna(subset=['mean_train_sec']).sort_values('mean_train_sec').iloc[0]
        presentation_lines.append(
            f"- 시간 기준으로는 `{fastest_row['recipe_name']}`가 평균 {fastest_row['mean_train_sec']:.1f}초로 가장 빨랐습니다."
        )

    if failure_df.empty:
        presentation_lines.append('- 이번 실행에서는 치명적인 실패 없이 기준 결과를 확보할 수 있었습니다.')
    else:
        top_issue_label = failure_df['issue_type'].fillna('기타 실행 이슈').value_counts().index[0]
        presentation_lines.append(
            f"- 실패는 총 {len(failure_df)}건 있었고, 가장 많이 관찰된 문제 유형은 `{top_issue_label}`였습니다."
        )

    presentation_lines.append('- 이 결과는 다른 모델과 비교할 때 AutoGluon GPU 기본 설정의 기준 점수로 사용할 수 있습니다.')
    presentation_lines.append('- 또 GPU를 켠다고 모든 단계가 GPU로 도는 것은 아니므로, 로그와 실패 유형을 함께 확인하는 것이 중요합니다.')

    display(Markdown('\n'.join(presentation_lines)))


Raw experiment results


,task_order,dataset,recipe_name,recipe_description,score,train_sec,best_model,trained_models,status,issue_type,error
0,10,guillermo,gpu_default,"GPU enabled, default search space",0.905757,611.62,WeightedEnsemble_L2,8,ok,None,
1,11,riccardo,gpu_default,"GPU enabled, default search space",0.999734,605.26,WeightedEnsemble_L2,5,ok,None,
2,12,amazon_employee,gpu_default,"GPU enabled, default search space",0.883654,96.66,WeightedEnsemble_L2,12,ok,None,
3,13,nomao,gpu_default,"GPU enabled, default search space",0.996191,165.55,WeightedEnsemble_L2,12,ok,None,
4,14,bank-marketing,gpu_default,"GPU enabled, default search space",0.936927,167.88,WeightedEnsemble_L2,12,ok,None,
5,15,adult,gpu_default,"GPU enabled, default search space",0.931600,175.66,WeightedEnsemble_L2,12,ok,None,
6,16,kddcup09_appetency,gpu_default,"GPU enabled, default search space",0.859398,199.70,WeightedEnsemble_L2,12,ok,None,
7,17,apsfailure,gpu_default,"GPU enabled, default search space",0.996174,298.76,WeightedEnsemble_L2,12,ok,None,
8,18,numerai28.6,gpu_default,"GPU enabled, default search space",0.522821,252.54,WeightedEnsemble_L2,12,ok,None,


Recipe summary for presentation


,recipe_name,recipe_description,planned_runs,success_runs,failed_runs,mean_score,median_score,mean_train_sec,median_train_sec,mean_trained_models,success_rate
0,gpu_default,"GPU enabled, default search space",9,9,0,0.892473,0.9316,285.958889,199.7,10.777778,1.0


Score pivot


recipe_name,gpu_default
dataset,
adult,0.931600
amazon_employee,0.883654
apsfailure,0.996174
bank-marketing,0.936927
guillermo,0.905757
kddcup09_appetency,0.859398
nomao,0.996191
numerai28.6,0.522821
riccardo,0.999734


Observed failures: none


## Presentation Draft
- 이번 실험은 OpenML 9개 태스크를 Kaggle T4, 600초, train 80% / test 20% 조건에서 실행한 결과입니다.
- 이번 노트북은 AutoGluon 내부 recipe 비교 대신 `gpu_default` 설정 하나를 기준 결과로 사용합니다.
- 평균 점수 기준으로는 `gpu_default`가 가장 높았고, 평균 점수는 0.8925였습니다.
- 시간 기준으로는 `gpu_default`가 평균 286.0초로 가장 빨랐습니다.
- 이번 실행에서는 치명적인 실패 없이 기준 결과를 확보할 수 있었습니다.
- 이 결과는 다른 모델과 비교할 때 AutoGluon GPU 기본 설정의 기준 점수로 사용할 수 있습니다.
- 또 GPU를 켠다고 모든 단계가 GPU로 도는 것은 아니므로, 로그와 실패 유형을 함께 확인하는 것이 중요합니다.